In [2]:
# =========================
# =========================
# COMP3010 BLEVE — FULL NOTEBOOK PIPELINE (Dev Rewrite)
# Key decisions:
# - Train on log1p(y) (stability + better relative error behavior)
# - Predict with expm1
# - Leakage-safe GroupKFold using scenario-level grouping (exclude sensor-specific fields)
# - Robust encoding + imputation
# - Stable MAPE implementation + group sanity checks
# =========================


# -------------------------
# CELL 0 — CONFIG
# -------------------------
import os
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5

DATA_DIR = "."  # change if needed
TRAIN_PATH  = os.path.join(DATA_DIR, "train.csv")
TEST_PATH   = os.path.join(DATA_DIR, "test.csv")
SAMPLE_PATH = os.path.join(DATA_DIR, "sample_prediction.csv")

OUT_PATH = "prediction.csv"

np.random.seed(SEED)


In [3]:
# =========================
# CELL 2 — LOAD DATA (LOCAL: ../data folder)
# =========================

DATA_DIR = "."

print("Current working dir:", os.getcwd())
print("Files in DATA_DIR:", os.listdir(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_prediction.csv")

print("Loaded shapes:")
print("train :", train.shape)
print("test  :", test.shape)
print("sample:", sample.shape)

Current working dir: C:\Users\ibrah\Untitled Folder 1
Files in DATA_DIR: ['.ipynb_checkpoints', '1st.ipynb', 'B_P_M2_previous.ipynb', 'B_P_P_M2.ipynb', 'COMP3010_Assignment_Tri1_2026-1.pdf', 'model.ipynb', 'prediction.csv', 'sample_prediction.csv', 'submission.csv', 'test.csv', 'train.csv', 'Untitled.ipynb', 'weeew.ipynb']
Loaded shapes:
train : (10050, 25)
test  : (3203, 24)
sample: (3203, 2)


In [4]:
# -------------------------
# CELL 2 — CLEAN + NORMALIZE COLUMN NAMES
# -------------------------
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in ["Unnamed: 0", "index"]:
        if c in df.columns and df[c].nunique() == len(df):
            df = df.drop(columns=[c])
    return df

def normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in ["Status", "Sensor Position Side"]:
        if c in df.columns:
            s = df[c].astype("string").str.strip().str.lower()
            s = s.replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
            df[c] = s
    return df

train = normalize_cols(train)
test  = normalize_cols(test)

train = drop_index_cols(train)
test  = drop_index_cols(test)

train = normalize_cats(train)
test  = normalize_cats(test)

train = train.drop_duplicates()
print("After clean:", train.shape, test.shape)

After clean: (10000, 25) (3203, 23)


In [5]:
# -------------------------
# CELL 3 — FEATURE ENGINEERING (safe + consistent)
# Also renames columns to avoid whitespace warnings in LightGBM
# -------------------------
def safe_div(a, b):
    b = np.where(b == 0, np.nan, b)
    return a / b

def fe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # rename spaces -> underscores once for the whole pipeline
    df.columns = df.columns.str.replace(r"\s+", "_", regex=True)

    def has(cols): return all(c in df.columns for c in cols)

    # tank volume + shape
    if has(["Tank_Width_(m)", "Tank_Length_(m)", "Tank_Height_(m)"]):
        W = df["Tank_Width_(m)"].astype(float)
        L = df["Tank_Length_(m)"].astype(float)
        H = df["Tank_Height_(m)"].astype(float)
        df["tank_vol"] = W * L * H
        df["tank_w_over_l"] = safe_div(W, L)

    # vapour fraction
    if has(["Vapour_Height_(m)", "Tank_Height_(m)"]):
        df["vapour_height_frac"] = safe_div(df["Vapour_Height_(m)"].astype(float),
                                            df["Tank_Height_(m)"].astype(float))

    # obstacle area
    if has(["Obstacle_Width_(m)", "Obstacle_Height_(m)"]):
        df["obs_area"] = df["Obstacle_Width_(m)"].astype(float) * df["Obstacle_Height_(m)"].astype(float)

    # sensor radius
    if has(["Sensor_Position_x_(m)", "Sensor_Position_y_(m)", "Sensor_Position_z_(m)"]):
        x = df["Sensor_Position_x_(m)"].astype(float)
        y = df["Sensor_Position_y_(m)"].astype(float)
        z = df["Sensor_Position_z_(m)"].astype(float)
        df["sensor_r"] = np.sqrt(x*x + y*y + z*z)

    # sensor_r / obstacle distance
    if has(["sensor_r", "Obstacle_Distance_to_BLEVE_(m)"]):
        df["sensor_r_over_obsdist"] = safe_div(df["sensor_r"].astype(float),
                                               df["Obstacle_Distance_to_BLEVE_(m)"].astype(float))

    # hard safety
    df = df.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    return df

train_fe = fe(train)
test_fe  = fe(test)

# target auto-detect (since we renamed spaces->underscores)
if "Target_Pressure_(bar)" in train_fe.columns:
    TARGET = "Target_Pressure_(bar)"
elif "Target Pressure (bar)" in train_fe.columns:
    TARGET = "Target Pressure (bar)"
else:
    raise KeyError("Target column not found after FE.")

print("Using TARGET:", TARGET)
print("FE shapes:", train_fe.shape, test_fe.shape)

Using TARGET: Target_Pressure_(bar)
FE shapes: (10000, 29) (3203, 27)


In [13]:
# -------------------------
# CELL 4 — SCENARIO GROUPS (leakage-safe)
# Group key must use ONLY scenario/config variables (exclude sensor fields + target)
# -------------------------
SENSOR_FIELDS = [
    "Sensor_ID",
    "Sensor_Position_Side",
    "Sensor_Position_x_(m)",
    "Sensor_Position_y_(m)",
    "Sensor_Position_z_(m)",
    "sensor_r",
    "sensor_r_over_obsdist",
]

exclude_for_group = set(SENSOR_FIELDS + [TARGET])

scenario_cols = [c for c in train_fe.columns if c not in exclude_for_group]

gdf = train_fe[scenario_cols].copy()

# stabilize numeric representation
for c in gdf.columns:
    if pd.api.types.is_numeric_dtype(gdf[c]):
        gdf[c] = gdf[c].round(6)
    gdf[c] = gdf[c].astype(str)

groups = pd.util.hash_pandas_object(gdf, index=False).astype("int64").to_numpy()

u, cnt = np.unique(groups, return_counts=True)
print("unique groups:", len(u), "rows:", len(groups))
print("group size min/median/max:", int(cnt.min()), float(np.median(cnt)), int(cnt.max()))

Using knob cols: ['Tank_Width_(m)', 'Tank_Length_(m)', 'Tank_Height_(m)', 'Vapour_Height_(m)', 'Obstacle_Width_(m)', 'Obstacle_Height_(m)', 'Obstacle_Distance_to_BLEVE_(m)', 'Status', 'Sensor_Position_Side']
unique groups: 9937 rows: 10000
group size min/median/max: 1 1.0 2
['Unnamed:_0', 'Tank_Failure_Pressure_(bar)', 'Liquid_Ratio', 'Tank_Width_(m)', 'Tank_Length_(m)', 'Tank_Height_(m)', 'BLEVE_Height_(m)', 'Vapour_Height_(m)', 'Vapour_Temperature_(K)', 'Liquid_Temperature_(K)', 'Obstacle_Distance_to_BLEVE_(m)', 'Obstacle_Width_(m)', 'Obstacle_Height_(m)', 'Obstacle_Thickness_(m)', 'Obstacle_Angle', 'Status', 'Liquid_Critical_Pressure_(bar)', 'Liquid_Boiling_Temperature_(K)', 'Liquid_Critical_Temperature_(K)', 'Sensor_ID', 'Sensor_Position_Side', 'Sensor_Position_x', 'Sensor_Position_y', 'Sensor_Position_z', 'Target_Pressure_(bar)', 'tank_vol', 'tank_w_over_l', 'vapour_height_frac', 'obs_area']
(10000, 29)
   Unnamed:_0  Tank_Failure_Pressure_(bar)  Liquid_Ratio  Tank_Width_(m)  Tank

In [7]:
# -------------------------
# CELL 5 — BUILD X/y (align + encode + constants)
# Train on log1p(y)
# -------------------------
y = train_fe[TARGET].astype(float).to_numpy()
y = np.clip(y, 1e-6, None)
y_log = np.log1p(y)

X_train = train_fe.drop(columns=[TARGET]).copy()
X_test  = test_fe.copy()

# align columns
common = X_train.columns.intersection(X_test.columns)
X_train = X_train[common].copy()
X_test  = X_test[common].copy()

# encode any object/string cols
obj_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols:
    X_train[c] = X_train[c].fillna("NA").astype(str).str.strip().str.lower()
    X_test[c]  = X_test[c].fillna("NA").astype(str).str.strip().str.lower()

    cats = pd.Index(pd.concat([X_train[c], X_test[c]], axis=0).unique())
    mapper = {k: i for i, k in enumerate(cats)}
    X_train[c] = X_train[c].map(mapper).astype("int32")
    X_test[c]  = X_test[c].map(mapper).astype("int32")

# final safety
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# drop constant columns
const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
if const_cols:
    X_train = X_train.drop(columns=const_cols)
    X_test  = X_test.drop(columns=const_cols)

print("X shapes:", X_train.shape, X_test.shape)
print("encoded obj cols:", len(obj_cols), "dropped constants:", len(const_cols))

X shapes: (10000, 27) (3203, 27)
encoded obj cols: 2 dropped constants: 0


In [8]:
# -------------------------
# -------------------------
# CELL 6 — CV (GroupKFold) + RandomForest (seed-weighted bagging) + stable MAPE
# -------------------------
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

def stable_mape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    y_true = np.clip(y_true, eps, None)
    y_pred = np.clip(y_pred, eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)))

def _clip_pred(p, eps=1e-6):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=eps, posinf=eps, neginf=eps)
    return np.clip(p, eps, None)

# ---- sanity ----
assert len(X_train) == len(y_log) == len(groups)
bad = X_train.select_dtypes(include=["object"]).columns.tolist()
if bad:
    raise ValueError(f"Object dtype columns found in X_train: {bad}")

gkf = GroupKFold(n_splits=N_SPLITS)

# ---- base params ----
base_params_rf = dict(
    n_estimators=1000,          # RF doesn't early-stop; use enough trees
    max_features=0.33,          # ~sqrt analog for regression (tune: 0.2–0.6)
    max_depth=None,             # fully grown trees (RF's default strength)
    min_samples_leaf=8,         # analog of min_data_in_leaf; regularizes
    min_samples_split=16,
    max_samples=0.85,           # row subsampling (like subsample=0.85)
    bootstrap=True,
    n_jobs=-1,
)

SEEDS = [42, 202, 1337, 3141]

# ---- per-seed CV + OOF preds ----
seed_fold_scores = {s: [] for s in SEEDS}
seed_oof = {s: np.zeros(len(X_train), dtype=float) for s in SEEDS}

for fold, (tr, va) in enumerate(gkf.split(X_train, y_log, groups), 1):
    Xtr, Xva = X_train.iloc[tr], X_train.iloc[va]
    ytr, yva = y_log[tr], y_log[va]

    for s in SEEDS:
        params = dict(base_params_rf)
        params["random_state"] = int(s)

        model = RandomForestRegressor(**params)
        model.fit(Xtr, ytr)

        pred_log = model.predict(Xva)
        pred = _clip_pred(np.expm1(pred_log))

        m = stable_mape(y[va], pred, eps=1e-6)
        seed_fold_scores[s].append(m)
        seed_oof[s][va] = pred

    # fold summary (seed-wise)
    msg = [f"s{s%1000:03d}:{seed_fold_scores[s][-1]:.5f}" for s in SEEDS]
    print(f"Fold {fold} | " + "  ".join(msg))

# ---- compute seed weights from mean CV (lower MAPE = higher weight) ----
seed_means = np.array([np.mean(seed_fold_scores[s]) for s in SEEDS], dtype=float)
seed_stds  = np.array([np.std(seed_fold_scores[s]) for s in SEEDS], dtype=float)

temp = 0.02
w_raw = np.exp(-(seed_means - seed_means.min()) / temp)
seed_weights = w_raw / w_raw.sum()

print("\nPer-seed CV mean/std + weights:")
for s, m, sd, w in zip(SEEDS, seed_means, seed_stds, seed_weights):
    print(f"  seed={s} | mean={m:.6f} std={sd:.6f} | weight={w:.4f}")

# ---- weighted OOF ----
oof = np.zeros(len(X_train), dtype=float)
for w, s in zip(seed_weights, SEEDS):
    oof += w * seed_oof[s]

fold_scores = []
for fold, (_, va) in enumerate(gkf.split(X_train, y_log, groups), 1):
    m = stable_mape(y[va], oof[va], eps=1e-6)
    fold_scores.append(m)
    print(f"Weighted Fold {fold} | stable MAPE: {m:.5f}")

print("\nWEIGHTED CV stable MAPE mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

Fold 1 | s042:130.07328  s202:130.14857  s337:129.28292  s141:130.49349
Fold 2 | s042:82.30607  s202:84.70940  s337:83.42867  s141:81.37460
Fold 3 | s042:137.32552  s202:135.20457  s337:139.52700  s141:134.91888
Fold 4 | s042:101.83980  s202:98.71012  s337:101.61658  s141:108.74490
Fold 5 | s042:0.24396  s202:0.24371  s337:0.24381  s141:0.24377

Per-seed CV mean/std + weights:
  seed=42 | mean=90.357725 std=49.194425 | weight=0.0000
  seed=202 | mean=89.803274 std=48.611142 | weight=1.0000
  seed=1337 | mean=90.819795 std=49.451013 | weight=0.0000
  seed=3141 | mean=91.155129 std=49.259937 | weight=0.0000
Weighted Fold 1 | stable MAPE: 130.14857
Weighted Fold 2 | stable MAPE: 84.70940
Weighted Fold 3 | stable MAPE: 135.20457
Weighted Fold 4 | stable MAPE: 98.71012
Weighted Fold 5 | stable MAPE: 0.24371

WEIGHTED CV stable MAPE mean: 89.80327400894785 std: 48.61114197795316


In [11]:
# -------------------------
# CELL 7 — TRAIN FULL + PREDICT TEST + SAVE  (weighted seed-bagged RF)
# Uses: X_train, X_test, y_log, sample, OUT_PATH, SEEDS
# Uses from CELL 6: base_params_rf, seed_weights
# -------------------------
import numpy as np
from sklearn.ensemble import RandomForestRegressor

def _clip_pred(p, eps=1e-6):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=eps, posinf=eps, neginf=eps)
    return np.clip(p, eps, None)

# guardrails
bad = X_train.select_dtypes(include=["object"]).columns.tolist()
if bad:
    raise ValueError(f"Object dtype columns found in X_train: {bad}")
if "base_params_rf" not in globals():
    raise NameError("base_params_rf not found. Run CELL 6 first.")
if "seed_weights" not in globals():
    raise NameError("seed_weights not found. Run CELL 6 first.")
if len(seed_weights) != len(SEEDS):
    raise ValueError("seed_weights length must match SEEDS.")

test_pred = None
for w, s in zip(seed_weights, SEEDS):
    p = dict(base_params_rf)
    p["random_state"] = int(s)

    final = RandomForestRegressor(**p)
    final.fit(X_train, y_log)

    pred_log = final.predict(X_test)
    pred = _clip_pred(np.expm1(pred_log))

    if test_pred is None:
        test_pred = w * pred
    else:
        test_pred += w * pred

test_pred = _clip_pred(test_pred)
sub = sample.copy()
sub.iloc[:, 1] = test_pred
sub.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
sub.head()

Saved: prediction.csv


,ID,Target Pressure (bar)
0,0,0.104736
1,1,0.104712
2,2,0.104879
3,3,0.077306
4,4,0.075215
